# 📊 Batch Training Dashboard

Single pane of glass for monitoring all experiments.
Re-run this notebook at any time to refresh status.

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime

# Environment-aware root detection
if Path('/content/drive/MyDrive').exists():
    PROJECT_ROOT = Path('/content/drive/MyDrive/repos/deep-learning')
else:
    PROJECT_ROOT = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'

# ── Scan completed.json markers ──
experiments = []
for exp_dir in sorted(EXPERIMENTS_DIR.iterdir()):
    if not exp_dir.is_dir() or exp_dir.name.startswith('_'):
        continue
    marker = exp_dir / 'completed.json'
    if marker.exists():
        data = json.loads(marker.read_text())
        data['_dir'] = exp_dir.name
        experiments.append(data)
    else:
        experiments.append({
            '_dir': exp_dir.name,
            'experiment': exp_dir.name,
            'success': None,
            'completed_at': None,
            'duration_seconds': None,
            'best_val_acc': None,
            'model': '—',
            'error': None,
        })

print(f'Found {len(experiments)} experiments ({sum(1 for e in experiments if e["success"] is True)} completed, '
      f'{sum(1 for e in experiments if e["success"] is False)} failed, '
      f'{sum(1 for e in experiments if e["success"] is None)} pending)\n')

In [ ]:
# ── Dashboard Table ──
STATUS_MAP = {True: '✅ Done', False: '❌ Failed', None: '⏳ Pending'}

header = f"{'Experiment':<30} {'Status':<12} {'Best Acc':>10} {'Duration':>10} {'Model':<15} {'Completed At':<20}"
print(header)
print('─' * len(header))

for e in experiments:
    name = e['_dir']
    status = STATUS_MAP.get(e['success'], '❓')
    acc = f"{e['best_val_acc']:.4f}" if e['best_val_acc'] is not None else '—'
    dur = f"{e['duration_seconds']/60:.1f}m" if e['duration_seconds'] is not None else '—'
    model = e.get('model', '—') or '—'
    completed = e.get('completed_at', '') or '—'
    if completed != '—' and len(completed) > 19:
        completed = completed[:19]
    print(f"{name:<30} {status:<12} {acc:>10} {dur:>10} {model:<15} {completed:<20}")

# Summary stats
done = [e for e in experiments if e['success'] is True]
if done:
    best = max(done, key=lambda x: x['best_val_acc'] or 0)
    total_time = sum(e['duration_seconds'] for e in done if e['duration_seconds'])
    print(f"\n🏆 Best overall: {best['_dir']} ({best['best_val_acc']:.4f})")
    print(f"⏱️  Total training time: {total_time/60:.1f}m ({total_time/3600:.1f}h)")

In [ ]:
# ── MLflow Cross-Experiment Comparison ──
try:
    from src.config.paths import setup_mlflow, MLFLOW_TRACKING_URI
    mlflow = setup_mlflow()
    
    print(f'MLflow tracking: {MLFLOW_TRACKING_URI}\n')
    
    all_experiments = mlflow.search_experiments()
    dl_experiments = [exp for exp in all_experiments if exp.name != 'Default']
    
    if not dl_experiments:
        print('⚠️ No MLflow experiments found yet.')
    else:
        print(f"{'MLflow Experiment':<35} {'Runs':>5} {'Best Val Acc':>12} {'Last Run':<20}")
        print('─' * 75)
        
        for exp in sorted(dl_experiments, key=lambda x: x.name):
            runs = mlflow.search_runs(
                experiment_ids=[exp.experiment_id],
                order_by=['start_time DESC'],
                max_results=100
            )
            if runs.empty:
                print(f"{exp.name:<35} {'0':>5} {'—':>12} {'—':<20}")
                continue
            
            n_runs = len(runs)
            best_acc = runs['metrics.best_val_acc'].max() if 'metrics.best_val_acc' in runs.columns else None
            best_str = f"{best_acc:.4f}" if best_acc and not str(best_acc) == 'nan' else '—'
            last_run = str(runs['start_time'].iloc[0])[:19] if 'start_time' in runs.columns else '—'
            print(f"{exp.name:<35} {n_runs:>5} {best_str:>12} {last_run:<20}")
        
        print(f"\nℹ️  Total: {len(dl_experiments)} experiments, {sum(len(mlflow.search_runs(experiment_ids=[e.experiment_id])) for e in dl_experiments)} runs")

except ImportError:
except Exception as e:
    print(f'⚠️ MLflow query failed: {e}')

In [ ]:
# ── Failed Experiments Detail ──
failed = [e for e in experiments if e['success'] is False]
if failed:
    print(f'\n❌ {len(failed)} Failed Experiments:\n')
    for e in failed:
        print(f"  {e['_dir']}")
        print(f"    Error: {e.get('error', 'unknown')}")
        print(f"    Time:  {e['duration_seconds']/60:.1f}m" if e['duration_seconds'] else '')
        print()
else:
    print('\n✅ No failed experiments!')